## ① 설정
수집에 필요한 값을 정하는 셀입니다.*
- `API_KEY`: OpenDART 인증키 (노트북을 공유할 때는 지울 것)
- `OUT_DIR`: 결과 저장 폴더 (`dart_data`)
- `YEARS`: 2017–2023. 2017은 전기 자산·평균 차입금 계산용
- `FS_DIV`: `OFS` = 별도재무제표(기본), `CFS` = 연결
- `REPRT`: `11011` = 사업보고서(연간)
- `SLEEP`, `MAX_CALLS_PER_RUN`: 호출 속도와 하루 호출 한도(약 2만 건) 보호
- `MANUF_KSIC2`: 제조업 KSIC 중분류 10~34
- `DEBT_ABSENT_AS_ZERO`: 차입 계정 줄이 아예 없으면 0으로 볼지 (가정이므로 강건성에서 False로도 확인)


In [20]:
API_KEY  = "2a4271caaef7404ab229e5b8012a639363f2f4b1"     # OpenDART 인증키 (이 노트북을 공유할 땐 지우세요)
OUT_DIR  = "dart_data"                 # 결과 저장 폴더 (없으면 생성)
YEARS    = list(range(2017, 2024))     # 2017은 전기 자산·평균 차입금 계산용
FS_DIV   = "OFS"                       # OFS=별도(권장), CFS=연결
REPRT    = "11011"                     # 사업보고서(연간)
SLEEP    = 0.15                        # 호출 사이 대기(초). 오류가 잦으면 0.3~0.5로
MAX_CALLS_PER_RUN = 19000              # 한 번 실행에서 쓸 최대 호출 수(일 한도 보호)
MANUF_KSIC2 = [f"{i:02d}" for i in range(10, 35)]   # 제조업 KSIC 중분류 10~34
DEBT_ABSENT_AS_ZERO = True 

## ② 준비: 라이브러리와 API 호출 함수
- 필요한 라이브러리를 불러오고 저장 폴더(`dart_data`, `dart_data/raw`)를 만듭니다.
- `_get()`: OpenDART에 요청을 보냅니다. 네트워크 오류가 나면 최대 5번 재시도합니다.
- `get_json()`: 응답의 상태코드를 확인합니다. 인증키 오류(010·011), 한도 초과(020) 등은 계속해도 소용없으므로 **즉시 멈춥니다.** `013`(데이터 없음)은 정상적으로 기록하고 넘어갑니다.
- 셀 3~5에서 모두 이 함수를 쓰므로, 노트북을 다시 열면 **셀 1·2부터 실행**해야 합니다.

In [21]:
import os, io, re, time, zipfile, json
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import requests
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **k): return x

BASE = "https://opendart.fss.or.kr/api"
os.makedirs(OUT_DIR, exist_ok=True)
RAW_DIR = os.path.join(OUT_DIR, "raw"); os.makedirs(RAW_DIR, exist_ok=True)
SESSION = requests.Session()
CALLS = {"n": 0}

STOP_STATUS = {"010": "등록되지 않은 키", "011": "사용할 수 없는 키", "012": "접근할 수 없는 IP",
               "020": "요청 제한 초과(일 한도)", "800": "시스템 점검", "901": "키 보유기간 만료"}

class StopAll(Exception):
    pass

def _get(endpoint, **params):
    if CALLS["n"] >= MAX_CALLS_PER_RUN:
        raise StopAll(f"이번 실행의 호출 상한 {MAX_CALLS_PER_RUN}건 도달 → 내일 같은 셀을 다시 실행하세요")
    url = f"{BASE}/{endpoint}"
    for attempt in range(5):
        try:
            r = SESSION.get(url, params={"crtfc_key": API_KEY, **params}, timeout=30)
            r.raise_for_status()
            CALLS["n"] += 1
            time.sleep(SLEEP)
            return r
        except requests.RequestException as e:
            wait = 2 ** attempt
            print(f"  네트워크 오류({e.__class__.__name__}), {wait}초 후 재시도")
            time.sleep(wait)
    raise StopAll("네트워크 오류가 5회 연속 → 인터넷 연결 확인 후 다시 실행")

def get_json(endpoint, **params):
    j = _get(endpoint, **params).json()
    st = j.get("status")
    if st in STOP_STATUS:
        raise StopAll(f"DART 상태 {st}: {STOP_STATUS[st]} / {j.get('message')}")
    return st, j

print("준비 완료. 인증키 앞 4자리:", API_KEY[:4] + "****")

준비 완료. 인증키 앞 4자리: 2a42****


## ③ 회사 목록(고유번호) 받기
- OpenDART의 **고유번호(corpCode.xml)** API로 DART에 등록된 전체 회사 목록을 받습니다. 호출 1건입니다.
- 회사마다 8자리 `corp_code`가 있고, 이후 모든 API는 이 번호로 조회합니다.
- 종목코드(`stock_code`)가 있는 회사, 즉 상장사만 남깁니다.
- 결과는 `corp_codes.csv`로 저장되며, 다시 실행하면 저장된 파일을 읽습니다(호출 0건).

In [5]:
corp_path = os.path.join(OUT_DIR, "corp_codes.csv")
if os.path.exists(corp_path):
    corps = pd.read_csv(corp_path, dtype=str).fillna("")
else:
    r = _get("corpCode.xml")
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        root = ET.fromstring(z.read(z.namelist()[0]))
    corps = pd.DataFrame([{c.tag: (c.text or "").strip() for c in item} for item in root.iter("list")])
    corps.to_csv(corp_path, index=False, encoding="utf-8-sig")
listed = corps[corps["stock_code"].str.strip().ne("")].copy()
print(f"전체 {len(corps):,}개 회사 중 상장(종목코드 있음) {len(listed):,}개")

전체 119,894개 회사 중 상장(종목코드 있음) 4,010개


## ④ 기업개황 → 제조업·12월 결산 기업 고르기
- 상장사마다 **기업개황(company.json)** API를 호출해 업종코드(`induty_code`, KSIC)와 결산월(`acc_mt`)을 받습니다. 상장사 수만큼 호출하므로 시간이 걸립니다.
- 업종코드 앞 2자리가 10~34(제조업)이고 결산월이 12월인 회사만 남깁니다.
- 100개마다 `company_info.csv`에 저장하므로, **끊겨도 다시 실행하면 이어서 받습니다.**
- 대상 목록은 `firms_manuf_dec.csv`로 저장되고, 마지막에 "재무제표 호출 예정 ○건"이 출력됩니다.

In [6]:
info_path = os.path.join(OUT_DIR, "company_info.csv")
done = pd.read_csv(info_path, dtype=str).fillna("") if os.path.exists(info_path) else pd.DataFrame(columns=["corp_code"])
todo = listed[~listed["corp_code"].isin(done["corp_code"])]
print(f"기업개황: 받은 {len(done):,}개 / 남은 {len(todo):,}개")

buf = []
def flush():
    global done, buf
    if buf:
        done = pd.concat([done, pd.DataFrame(buf)], ignore_index=True)
        done.to_csv(info_path, index=False, encoding="utf-8-sig")
        buf = []
try:
    for cc in tqdm(todo["corp_code"], desc="기업개황"):
        st, j = get_json("company.json", corp_code=cc)
        keep = ["corp_name", "stock_code", "corp_cls", "induty_code", "acc_mt", "est_dt"]
        buf.append({"corp_code": cc, "status": st, **{k: j.get(k, "") for k in keep}})
        if len(buf) >= 100:
            flush()
except StopAll as e:
    print("멈춤:", e)
finally:
    flush()

info = done.copy()
info["ksic2"] = info["induty_code"].fillna("").str[:2]
firms = info[info["ksic2"].isin(MANUF_KSIC2) & info["acc_mt"].fillna("").str.zfill(2).eq("12")].copy()
firms.to_csv(os.path.join(OUT_DIR, "firms_manuf_dec.csv"), index=False, encoding="utf-8-sig")
print(f"제조업·12월 결산 상장사 {len(firms):,}개 → 재무제표 호출 예정 {len(firms)*len(YEARS):,}건")

기업개황: 받은 4,010개 / 남은 0개


기업개황: 0it [00:00, ?it/s]

제조업·12월 결산 상장사 2,209개 → 재무제표 호출 예정 15,463건


## ⑤ 재무제표 다운로드
- 대상 회사 × 연도(2017–2023)마다 **단일회사 전체 재무제표(fnlttSinglAcntAll)** API를 호출합니다. 가장 오래 걸리는 셀입니다.
- 받은 재무제표(재무상태표·손익계산서·현금흐름표 전체)를 **원본 그대로** `raw/회사코드_연도_OFS.csv`로 저장합니다. 그래서 나중에 계정 추출 규칙을 바꿔도 다시 받을 필요가 없습니다.
- 회사-연도별 응답 상태(000 정상 / 013 데이터 없음)를 `download_log.csv`에 기록합니다. 끊기면 다시 실행해서 이어받습니다.

In [7]:
log_path = os.path.join(OUT_DIR, "download_log.csv")
log = pd.read_csv(log_path, dtype=str).fillna("") if os.path.exists(log_path) else pd.DataFrame(columns=["corp_code", "year", "fs_div", "status", "message"])
done_keys = set(zip(log["corp_code"], log["year"].astype(str), log["fs_div"]))
jobs = [(cc, str(y)) for cc in firms["corp_code"] for y in YEARS if (cc, str(y), FS_DIV) not in done_keys]
print(f"남은 작업 {len(jobs):,}건 (이미 처리 {len(done_keys):,}건)")

new_log = []
def flush_log():
    global log, new_log
    if new_log:
        log = pd.concat([log, pd.DataFrame(new_log)], ignore_index=True)
        log.to_csv(log_path, index=False, encoding="utf-8-sig")
        new_log = []
try:
    for cc, y in tqdm(jobs, desc="재무제표"):
        st, j = get_json("fnlttSinglAcntAll.json", corp_code=cc, bsns_year=y, reprt_code=REPRT, fs_div=FS_DIV)
        if st == "000":
            pd.DataFrame(j["list"]).to_csv(os.path.join(RAW_DIR, f"{cc}_{y}_{FS_DIV}.csv"), index=False, encoding="utf-8-sig")
        new_log.append({"corp_code": cc, "year": y, "fs_div": FS_DIV, "status": st, "message": j.get("message", "")})
        if len(new_log) >= 50:
            flush_log()
except StopAll as e:
    print("멈춤:", e)
finally:
    flush_log()
print("상태코드 분포:", log[log["fs_div"] == FS_DIV]["status"].value_counts().to_dict(), "| 이번 실행 호출", CALLS["n"])

남은 작업 15,463건 (이미 처리 0건)


재무제표:  36%|████████████████████████▎                                          | 5621/15463 [22:14<54:20,  3.02it/s]

  네트워크 오류(ConnectionError), 1초 후 재시도
  네트워크 오류(ConnectionError), 2초 후 재시도


재무제표: 100%|████████████████████████████████████████████████████████████████| 15463/15463 [1:26:55<00:00,  2.96it/s]

상태코드 분포: {'000': 9837, '013': 5626} | 이번 실행 호출 15463


## ⑥ 계정 추출 규칙 (3차 수정본)
원본 재무제표에서 필요한 계정을 어떻게 찾을지 정하는 셀입니다. 회사마다 계정명이 달라서 변수마다 아래 순서로 찾습니다.
1. 표준계정코드(`account_id`)로 찾기 (구 택소노미 `ifrs_` 접두어도 통일)
2. 계정명이 정확히 같은 줄
3. 계정명에 특정 단어가 들어간 줄 (제외어가 들어간 줄은 빼기)

- **차입 계정**(단기차입금 → 유동성장기부채 → 유동성사채 → 장기차입금 → 사채 → 리스부채)은 위에서부터 차례로 줄을 가져가고, **한 번 쓴 줄은 다음 변수가 못 씁니다.** "단기차입금및유동성장기부채" 같은 결합 계정의 중복 집계를 막기 위해서입니다.
- 사채, 유동성사채, 리스부채, 이자 지급은 해당하는 줄을 **모두 합산**합니다.
- 이자는 현금흐름표의 '이자의 지급'(주)과 손익계산서의 '금융비용'(보조)을 받습니다. API가 주석을 주지 않아서 이자비용 항목은 없습니다. '이자의 수취'는 제외합니다.
- 리스부채는 기본 차입에 넣지 않고 강건성용으로 따로 받습니다(IFRS 16, 2019년부터).

In [22]:
def norm(s):
    return re.sub(r"[\s\(\)\[\]·\.,ㆍ\-_:0-9]", "", str(s))

# 차입 계정은 위에서부터 차례로 줄을 차지하고, 한 번 쓴 줄은 다음 변수가 못 씀(중복 집계 방지)
RULES = {
  "assets":     dict(sj=["BS"], ids=["ifrs-full_Assets"], exact=["자산총계"]),
  "cur_assets": dict(sj=["BS"], ids=["ifrs-full_CurrentAssets"], exact=["유동자산"]),
  "inventory":  dict(sj=["BS"], ids=["ifrs-full_Inventories"], exact=["재고자산"]),
  "cur_liab":   dict(sj=["BS"], ids=["ifrs-full_CurrentLiabilities"], exact=["유동부채"]),
  "liab":       dict(sj=["BS"], ids=["ifrs-full_Liabilities"], exact=["부채총계"]),
  "equity":     dict(sj=["BS"], ids=["ifrs-full_Equity"], exact=["자본총계"]),
  "st_borrow":  dict(sj=["BS"], debt=True, ids=["ifrs-full_ShorttermBorrowings"],
                     exact=["단기차입금", "유동차입금", "단기차입부채"],
                     contains=["단기차입"], exclude=[]),   # 결합계정은 단기로 배정
  "cur_ltd":    dict(sj=["BS"], debt=True, ids=["ifrs-full_CurrentPortionOfLongtermBorrowings"],
                     exact=["유동성장기부채", "유동성장기차입금", "유동성장기차입부채"],
                     contains=["유동성장기", "유동성대체"], exclude=["리스", "미지급"]),
  "cur_bonds":  dict(sj=["BS"], debt=True, ids=["dart_CurrentPortionOfConvertibleBonds", "dart_CurrentPortionOfBondWithWarrant"],
                     exact=["유동성사채"],
                     contains=["유동성사채", "유동성전환사채", "유동성신주인수권부사채", "유동성교환사채"],
                     exclude=["비유동"], agg="sum"),
  "lt_borrow":  dict(sj=["BS"], debt=True, ids=["ifrs-full_LongtermBorrowings", "dart_LongTermBorrowingsGross"],
                     exact=["장기차입금", "장기차입부채", "비유동차입금"],
                     contains=["장기차입", "비유동차입"], exclude=["유동성", "대체"]),
  "bonds":      dict(sj=["BS"], debt=True, ids=[], exact=["사채", "장기사채", "비유동사채", "비유동성사채"],
                     contains=["사채"], exclude=["유동성", "단기", "할인", "할증", "상환"], agg="sum"),
  # 리스부채: 기본 차입에는 넣지 않고 강건성용으로만 따로 받음(IFRS16, 2019년부터)
  "lease":      dict(sj=["BS"], debt=True, ids=[], exact=[], contains=["리스부채"], exclude=["이자"], agg="sum"),
  "revenue":    dict(sj=["IS", "CIS"], ids=["ifrs-full_Revenue"], exact=["매출액", "수익매출액", "영업수익", "매출"]),
  "op_income":  dict(sj=["IS", "CIS"], ids=["dart_OperatingIncomeLoss"], exact=["영업이익", "영업이익손실", "영업손익"]),
  "fin_cost":   dict(sj=["IS", "CIS"], ids=["ifrs-full_FinanceCosts"], exact=["금융비용", "금융원가"]),
  "int_paid":   dict(sj=["CF"], ids=["ifrs-full_InterestPaidClassifiedAsOperatingActivities",
                                     "ifrs-full_InterestPaidClassifiedAsFinancingActivities"],
                     exact=["이자의지급", "이자지급", "이자지급액", "이자비용의지급", "이자지급영업", "이자지급재무"],
                     contains=["이자의지급", "이자지급", "이자비용의지급", "이자비용지급", "이자납부"],
                     exclude=["수취", "받은", "수입"], agg="sum", abs=True),
}
DEBT_VARS = ["st_borrow", "cur_ltd", "cur_bonds", "lt_borrow", "bonds"]

def to_num(s):
    s = str(s).replace(",", "").strip()
    if s.startswith("(") and s.endswith(")"):
        s = "-" + s[1:-1]
    if s in ("", "-", "nan", "None"):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

def pick(df, rule, used=None):
    used = used if used is not None else set()
    for sj in rule["sj"]:
        d = df[(df["sj_div"] == sj) & (~df.index.isin(used) if rule.get("debt") else True)]
        if d.empty:
            continue
        nm = d["account_nm"].map(norm)
        aid = d["account_id"].str.replace(r"^ifrs_", "ifrs-full_", regex=True)   # 구 택소노미 접두어 통일
        steps = [("id", aid.isin(rule.get("ids", []))),
                 ("exact", nm.isin([norm(x) for x in rule.get("exact", [])]))]
        if rule.get("contains"):
            m = nm.apply(lambda s: any(norm(k) in s for k in rule["contains"]))
            for ex in rule.get("exclude", []):
                m &= ~nm.str.contains(norm(ex), regex=False)
            steps.append(("contains", m))
        if rule.get("agg") == "sum":            # 합산 변수: 세 방식으로 찾은 줄을 모두 합침(한 줄은 한 번만)
            union = steps[0][1].copy()
            for _, mk in steps[1:]:
                union |= mk
            steps = [("union", union)]
        for how, mask in steps:
            hit = d[mask]
            if hit.empty:
                continue
            hit = hit.assign(_v=hit["thstrm_amount"].map(to_num)).dropna(subset=["_v"])
            if hit.empty:
                continue
            if "ord" in hit:
                hit = hit.assign(_o=pd.to_numeric(hit["ord"], errors="coerce")).sort_values("_o", kind="stable")
            if rule.get("agg") == "sum":
                v = hit["_v"].abs().sum() if rule.get("abs") else hit["_v"].sum()
                rows_used = set(hit.index)
            else:
                v = hit["_v"].iloc[0]
                v = abs(v) if rule.get("abs") else v
                rows_used = {hit.index[0]}
            names = " | ".join(hit.loc[list(rows_used), "account_nm"].astype(str).unique()[:3])
            return v, f"{sj}:{how}", names, rows_used
    return np.nan, "없음", "", set()

## ⑦ 계정 추출 실행
- `raw/` 폴더의 모든 재무제표에 셀 6의 규칙을 적용해 회사-연도별 값을 뽑습니다. API 호출은 없습니다.
- 변수마다 **어떤 방법으로, 어떤 계정명이 잡혔는지** `match_log.csv`에 기록합니다. 잘못 잡힌 계정을 찾을 때 씁니다.
- 마지막에 변수별 '없음'(못 찾음) 비율이 출력됩니다. 비율이 높은 변수는 셀 10의 진단표로 원인을 확인합니다.

In [23]:
rows, mlog = [], []
files = sorted(f for f in os.listdir(RAW_DIR) if f.endswith(f"_{FS_DIV}.csv"))
for f in tqdm(files, desc="계정 추출"):
    cc, y, _ = f[:-4].split("_")
    df = pd.read_csv(os.path.join(RAW_DIR, f), dtype=str).fillna("").reset_index(drop=True)
    rec = {"corp_code": cc, "year": int(y)}
    used = set()
    for var, rule in RULES.items():
        v, how, name, rows_used = pick(df, rule, used)
        if rule.get("debt"):
            used |= rows_used
        rec[var] = v
        mlog.append({"corp_code": cc, "year": int(y), "var": var, "how": how, "account_nm": name, "value": v})
    rows.append(rec)
wide = pd.DataFrame(rows)
match_log = pd.DataFrame(mlog)
match_log.to_csv(os.path.join(OUT_DIR, "match_log.csv"), index=False, encoding="utf-8-sig")
print(f"{len(wide):,}개 회사-연도 추출")
print("\n변수별 '없음' 비율(%)")
print((match_log.assign(miss=match_log["how"].eq("없음")).groupby("var")["miss"].mean() * 100).round(1).to_string())

계정 추출: 100%|███████████████████████████████████████████████████████████████████| 9837/9837 [20:13<00:00,  8.11it/s]


9,837개 회사-연도 추출

변수별 '없음' 비율(%)
var
assets         0.1
bonds         79.4
cur_assets     0.1
cur_bonds     78.8
cur_liab       0.2
cur_ltd       62.0
equity         0.0
fin_cost       1.1
int_paid       6.7
inventory      2.5
lease         52.7
liab           0.0
lt_borrow     43.2
op_income      0.0
revenue        0.2
st_borrow     28.5


## ⑧ 패널과 파생변수 만들기
> 노트북을 다시 열었다면 이 셀 전에 `firms = pd.read_csv(os.path.join(OUT_DIR, "firms_manuf_dec.csv"), dtype=str)`를 먼저 실행하세요.

- 회사 정보(이름, 업종코드)를 붙이고, 차입 계정 줄이 없으면 0으로 채웁니다(`*_absent`로 표시).
- 전기 자산·전기 차입금은 **연도가 연속일 때만** 붙입니다.
- 만드는 파생변수 (모두 근사치):
  - `E_st` = 단기차입금 / 자산 × 100 (**핵심 노출도**)
  - `E_st_cltd` = (단기차입 + 유동성장기부채 + 유동성사채) / 자산 × 100
  - `TB` = 총차입 / 자산 × 100,  `L` = 유동자산 / 유동부채 × 100
  - `ICR_paid` = 영업이익 / 이자의 지급 (배, 주),  `ICR_fc` = 영업이익 / 금융비용 (보조)
  - `r_eff` = 이자의 지급 / 평균 총차입 × 100 (실효 차입금리),  `r_eff_lease` = 리스부채 포함 버전
  - `IE_S` = 이자의 지급 / 매출 × 100
- 결과는 `panel_firm_year_OFS.csv`로 저장됩니다. **분석에 쓰는 파일입니다.**

In [24]:
firms = pd.read_csv(os.path.join(OUT_DIR, "firms_manuf_dec.csv"), dtype=str)
P = wide.merge(firms[["corp_code", "corp_name", "stock_code", "corp_cls", "induty_code", "ksic2"]], on="corp_code", how="left")
for v in DEBT_VARS:
    P[f"{v}_absent"] = P[v].isna() & P["assets"].notna()
    if DEBT_ABSENT_AS_ZERO:
        P.loc[P[f"{v}_absent"], v] = 0.0

P = P.sort_values(["corp_code", "year"])
P["borrow_total"] = P[DEBT_VARS].sum(axis=1, min_count=1)
P["borrow_short"] = P[["st_borrow", "cur_ltd", "cur_bonds"]].sum(axis=1, min_count=1)
g = P.groupby("corp_code")
P["assets_lag"] = g["assets"].shift(1)
P["borrow_total_lag"] = g["borrow_total"].shift(1)
P.loc[g["year"].diff().ne(1), ["assets_lag", "borrow_total_lag"]] = np.nan   # 연도가 연속일 때만 전기값

pos = lambda s: s.where(s > 0)
P["E_st"]       = 100 * P["st_borrow"] / pos(P["assets"])        # 단기차입금/자산
P["E_st_cltd"]  = 100 * P["borrow_short"] / pos(P["assets"])     # + 유동성장기부채·유동성사채
P["TB"]         = 100 * P["borrow_total"] / pos(P["assets"])     # 총차입/자산
P["L"]          = 100 * P["cur_assets"] / pos(P["cur_liab"])     # 유동비율
P["ICR_paid"]   = P["op_income"] / pos(P["int_paid"])            # 영업이익/이자의 지급(배, 주 근사)
P["ICR_fc"]     = P["op_income"] / pos(P["fin_cost"])            # 영업이익/금융비용(배, 보조)
P["ICR_paid_lt1"] = np.where(P["ICR_paid"].notna(), (P["ICR_paid"] < 1).astype(float), np.nan)
avg_b = (P["borrow_total"] + P["borrow_total_lag"]) / 2
P["r_eff"]      = 100 * P["int_paid"] / pos(avg_b)               # 실효 차입금리(%)
P["IE_S"]       = 100 * P["int_paid"] / pos(P["revenue"])        # 이자 지급/매출(%)

# 리스부채 포함 차입(강건성용). 리스 줄이 없으면 0 (2019년 이전엔 대부분 0)
P["lease"] = P["lease"].where(~(P["lease"].isna() & P["assets"].notna()), 0.0)
P["borrow_lease"] = P["borrow_total"] + P["lease"]
P["borrow_lease_lag"] = P.groupby("corp_code")["borrow_lease"].shift(1)
P.loc[P.groupby("corp_code")["year"].diff().ne(1), "borrow_lease_lag"] = np.nan
P["r_eff_lease"] = 100 * P["int_paid"] / pos((P["borrow_lease"] + P["borrow_lease_lag"]) / 2)

panel_path = os.path.join(OUT_DIR, f"panel_firm_year_{FS_DIV}.csv")
P.to_csv(panel_path, index=False, encoding="utf-8-sig")
print(f"저장: {panel_path}  ({P['corp_code'].nunique():,}개 회사, {len(P):,}행)")

저장: dart_data\panel_firm_year_OFS.csv  (1,633개 회사, 9,837행)


# 셀9

In [25]:
print("① 연도별 변수 확보율(%)")
cov = P.groupby("year")[["assets", "op_income", "revenue", "int_paid", "fin_cost", "st_borrow", "cur_liab"]].apply(lambda d: d.notna().mean() * 100)
cov.insert(0, "회사수", P.groupby("year")["corp_code"].nunique())
print(cov.round(1))

print("\n② 회계등식 점검: |자산 − (부채+자본)| / 자산 > 1% 인 회사-연도")
bad = P[(P["assets"] - P["liab"] - P["equity"]).abs() / P["assets"] > 0.01]
print(f"  {len(bad)}건", bad[["corp_name", "year"]].head(10).values.tolist())

print("\n③ 차입 계정이 없어 0으로 채운 비율(%)")
print({v: round(100 * P[f"{v}_absent"].mean(), 1) for v in DEBT_VARS})

print("\n④ '이자의 지급'을 못 찾은 회사-연도:", int(P["int_paid"].isna().sum()), "건 → match_log에서 계정명 확인")

print("\n⑤ 파생변수 분포")
print(P[["E_st", "E_st_cltd", "TB", "L", "ICR_paid", "r_eff", "IE_S"]].describe(percentiles=[.01, .5, .99]).T.round(2))

print("\n⑥ 계정 추출 확인용 표본 3개 회사(2022) ─ DART 사업보고서와 직접 대조")
s = P[P["year"] == 2022].sample(min(3, (P["year"] == 2022).sum()), random_state=1)
print(s[["corp_name", "assets", "st_borrow", "cur_ltd", "op_income", "int_paid", "fin_cost"]].to_string(index=False))

① 연도별 변수 확보율(%)
       회사수  assets  op_income  revenue  int_paid  fin_cost  st_borrow  \
year                                                                    
2017  1238    99.9      100.0     99.6      91.1      97.1       99.9   
2018  1302    99.8      100.0     99.8      90.6      97.4       99.9   
2019  1353    99.9      100.0     99.7      93.0      99.3      100.0   
2020  1413   100.0      100.0     99.9      94.6      99.6      100.0   
2021  1455    99.9       99.9     99.7      94.4      99.5       99.9   
2022  1497   100.0      100.0     99.9      94.4      99.5      100.0   
2023  1579    99.9      100.0     99.8      94.6      99.4       99.9   

      cur_liab  
year            
2017      99.6  
2018      99.7  
2019      99.9  
2020      99.8  
2021      99.8  
2022      99.9  
2023     100.0  

② 회계등식 점검: |자산 − (부채+자본)| / 자산 > 1% 인 회사-연도
  3건 [['(주)휴스토리', 2023], ['웅진에너지(주)', 2019], ['(주)세종메디칼', 2023]]

③ 차입 계정이 없어 0으로 채운 비율(%)
{'st_borrow': 28.4, 'cur_ltd': 61.9, 

# 셀10

In [26]:
used_map = {}
for r in match_log.itertuples():
    for nm in str(r.account_nm).split(" | "):
        if nm:
            used_map.setdefault((r.var, norm(nm)), 0)
var_of = {}
for (var, n) in used_map:
    var_of.setdefault(n, set()).add(var)

cnt = {}
for f in tqdm(files, desc="계정명 진단"):
    df = pd.read_csv(os.path.join(RAW_DIR, f), dtype=str, usecols=["sj_div", "account_id", "account_nm"]).fillna("")
    bs = df[(df["sj_div"] == "BS") & df["account_nm"].str.contains("차입|사채|금융부채|리스부채")]
    cf = df[(df["sj_div"] == "CF") & df["account_nm"].str.contains("이자")]
    for r in pd.concat([bs, cf]).drop_duplicates(["sj_div", "account_nm"]).itertuples():
        k = (r.sj_div, r.account_nm, r.account_id)
        cnt[k] = cnt.get(k, 0) + 1
diag = pd.DataFrame([dict(sj_div=k[0], account_nm=k[1], account_id=k[2], 회사연도수=v,
                          잡힌변수=",".join(sorted(var_of.get(norm(k[1]), [])))) for k, v in cnt.items()])
diag = diag.sort_values(["sj_div", "회사연도수"], ascending=[True, False])
diag.to_csv(os.path.join(OUT_DIR, "diag_account_names.csv"), index=False, encoding="utf-8-sig")
print(f"저장: diag_account_names.csv ({len(diag):,}개 계정명)")
print("\n안 잡힌 계정명 중 많이 나온 것 (상위 25)")
print(diag[diag["잡힌변수"].eq("")].head(25).to_string(index=False))

계정명 진단: 100%|█████████████████████████████████████████████████████████████████| 9837/9837 [04:07<00:00, 39.67it/s]

저장: diag_account_names.csv (1,251개 계정명)

안 잡힌 계정명 중 많이 나온 것 (상위 25)
sj_div              account_nm                                                                                          account_id  회사연도수 잡힌변수
    BS                기타유동금융부채                                                          ifrs-full_OtherCurrentFinancialLiabilities   2423     
    BS               기타비유동금융부채                                                       ifrs-full_OtherNoncurrentFinancialLiabilities   2267     
    BS                  기타금융부채                                                          ifrs-full_OtherCurrentFinancialLiabilities    776     
    BS                기타유동금융부채                                                               ifrs_OtherCurrentFinancialLiabilities    754     
    BS               기타비유동금융부채                                                            ifrs_OtherNoncurrentFinancialLiabilities    615     
    BS                  기타금융부채                                            